In [5]:
!pip install xgboost==1.7.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.3/200.3 MB 60.6 MB/s  0:00:03m0:00:0100:01


In [8]:
import joblib
import tarfile
import boto3
import os

# =========================
# CONFIGURACIÓN
# =========================
bucket_name = 'sisfall-fall-detection'

# Dónde está el archivo AHORA MISMO en S3 (lo que vemos en tu captura)
s3_key_origen = 'output/modelo_sisfall_optimizado.pkl' 

# Dónde lo vamos a guardar temporalmente en la máquina de SageMaker
ruta_modelo_local = 'modelo_sisfall_optimizado.pkl' 

# Nombres de salida
xgb_native_model = 'xgboost-model'
tar_name = 'model.tar.gz'
s3_key_destino = 'output/model.tar.gz'

# Inicializamos el cliente de S3
s3_client = boto3.client('s3')

# =========================
# 0. DESCARGAR EL MODELO DESDE S3 AL NOTEBOOK
# =========================
print("⬇️ Descargando modelo original desde S3...")
s3_client.download_file(bucket_name, s3_key_origen, ruta_modelo_local)
print("✔ Archivo .pkl descargado correctamente en la instancia")

# =========================
# 1. CARGAR MODELO .PKL
# =========================
modelo_cargado = joblib.load(ruta_modelo_local)
print("✔ Modelo .pkl cargado en memoria correctamente")

# =========================
# 2. EXPORTAR FORMATO NATIVO XGBOOST
# =========================
# Extraemos el motor puro (Booster) y lo guardamos
modelo_cargado.get_booster().save_model(xgb_native_model)

print("✔ Modelo exportado a formato nativo XGBoost")

# =========================
# 3. CREAR model.tar.gz
# =========================
with tarfile.open(tar_name, "w:gz") as tar:
    tar.add(xgb_native_model, arcname='xgboost-model')
print("✔ Archivo model.tar.gz creado correctamente")

# =========================
# 4. SUBIR A S3
# =========================
print("⬆️ Subiendo nuevo artefacto comprimido a S3...")
s3_client.upload_file(tar_name, bucket_name, s3_key_destino)

print(f"🚀 ¡TODO LISTO! Artefacto final subido a s3://{bucket_name}/{s3_key_destino}")

⬇️ Descargando modelo original desde S3...
✔ Archivo .pkl descargado correctamente en la instancia
✔ Modelo .pkl cargado en memoria correctamente
✔ Modelo exportado a formato nativo XGBoost
✔ Archivo model.tar.gz creado correctamente
⬆️ Subiendo nuevo artefacto comprimido a S3...
🚀 ¡TODO LISTO! Artefacto final subido a s3://sisfall-fall-detection/output/model.tar.gz
